# 08 — Malware Analysis Fundamentals

## What This Is
Malware analysis extracts indicators of compromise (IoCs), understands malware behaviour, and builds detection signatures. It splits into static analysis (examine code without running it) and dynamic analysis (observe runtime behaviour in a sandbox).

## Why It Matters
Malware analysts produce the YARA rules and detection logic that powers antivirus, EDR, and SIEM products. Every threat intelligence report — from ransomware groups to nation-state implants — begins with malware analysis.

## Where the Community Stands
Cuckoo Sandbox and ANY.RUN automate dynamic analysis. YARA is the universal language for static signatures. ML-based classifiers (Gradient Boosting on PE header features, graph neural nets for call graphs) augment manual analysis at scale.

## Authorized Testing Context
Malware reverse engineering is a defensive discipline conducted in isolated environments (isolated VMs, sandboxes). The code here simulates analysis of synthetic/hypothetical samples — no real malware is used or generated.

In [ ]:
import re
import hashlib
import struct
import random
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Set

# --- Synthetic binary samples (byte arrays, not real malware) ---
def make_synthetic_pe(embedded_strings=None, suspicious_apis=None):
    """Construct a synthetic PE-like byte blob for analysis simulation."""
    header = b'MZ' + b'\x90' * 60 + b'PE\x00\x00'  # DOS/PE signature
    apis = b'\x00'.join((a.encode() for a in (suspicious_apis or [])))
    strings_blob = b'\x00'.join((s.encode() for s in (embedded_strings or [])))
    padding = bytes(random.randint(0,255) for _ in range(200))
    return header + apis + strings_blob + padding

SAMPLE_A = make_synthetic_pe(
    embedded_strings=['http://c2.evil.example/beacon', 'cmd.exe', 'HKLM\\Run', 'powershell -enc'],
    suspicious_apis=['VirtualAllocEx','WriteProcessMemory','CreateRemoteThread','RegSetValueEx']
)
SAMPLE_B = make_synthetic_pe(
    embedded_strings=['192.168.1.100', 'notepad.exe', 'kernel32.dll'],
    suspicious_apis=['ReadFile','WriteFile','CreateFile']
)
SAMPLE_C = make_synthetic_pe(
    embedded_strings=['bitcoin:bc1q...', '.locked', 'YOUR FILES ARE ENCRYPTED', 'ransom.txt'],
    suspicious_apis=['CryptEncrypt','CryptAcquireContext','FindFirstFile','MoveFileEx']
)

for name, sample in [('Sample A', SAMPLE_A), ('Sample B', SAMPLE_B), ('Sample C', SAMPLE_C)]:
    md5 = hashlib.md5(sample).hexdigest()
    sha256 = hashlib.sha256(sample).hexdigest()
    print(f'{name}: {len(sample)} bytes  MD5={md5[:16]}...  SHA256={sha256[:16]}...')

In [ ]:
@dataclass
class StaticReport:
    sample_name: str
    md5: str
    sha256: str
    is_pe: bool
    extracted_strings: List[str]
    suspicious_strings: List[str]
    api_imports: List[str]
    suspicious_apis: List[str]
    iocs: Dict[str, List[str]]
    threat_categories: List[str]
    risk_score: int  # 0-100

# Suspicious API categories
INJECTION_APIS  = {'VirtualAllocEx','WriteProcessMemory','CreateRemoteThread','NtUnmapViewOfSection'}
CRYPTO_APIS     = {'CryptEncrypt','CryptDecrypt','CryptAcquireContext','BCryptEncrypt'}
PERSISTENCE_APIS= {'RegSetValueEx','RegCreateKeyEx','CreateService','SchtasksCreate'}
NETWORK_APIS    = {'InternetOpenUrl','WinHttpOpen','WSAConnect','send','recv'}
FILE_APIS       = {'FindFirstFile','FindNextFile','MoveFileEx','DeleteFileA'}

IOC_PATTERNS = {
    'url':     re.compile(r'https?://[\w./%-]+'),
    'ip':      re.compile(r'\b(?:\d{1,3}\.){3}\d{1,3}\b'),
    'domain':  re.compile(r'[a-zA-Z0-9-]+\.[a-zA-Z]{2,}(?:/[\w/%-]*)?' ),
    'btc':     re.compile(r'bc1[a-z0-9]{39,59}|[13][a-zA-Z0-9]{25,34}'),
    'registry':re.compile(r'HK[LCU][A-Z_]*\\[\w\\]+'),
}

def extract_printable_strings(data: bytes, min_len: int = 4) -> List[str]:
    return re.findall(rb'[\x20-\x7E]{' + str(min_len).encode() + rb',}', data)

SUSPICIOUS_KEYWORDS = [
    'powershell', 'cmd.exe', 'encrypt', 'locked', 'ransom',
    'bitcoin', 'YOUR FILES', 'c2.', '/beacon', 'HKLM',
]

def static_analyse(name: str, data: bytes) -> StaticReport:
    md5    = hashlib.md5(data).hexdigest()
    sha256 = hashlib.sha256(data).hexdigest()
    is_pe  = data[:2] == b'MZ'

    raw_strings = [s.decode('ascii', errors='ignore')
                   for s in extract_printable_strings(data)]

    susp_strings = [s for s in raw_strings
                    if any(kw.lower() in s.lower() for kw in SUSPICIOUS_KEYWORDS)]

    # API detection (look for known API names in strings)
    all_apis = set()
    all_api_names = INJECTION_APIS | CRYPTO_APIS | PERSISTENCE_APIS | NETWORK_APIS | FILE_APIS
    for s in raw_strings:
        if s in all_api_names:
            all_apis.add(s)
    susp_apis = list(all_apis)

    # IoC extraction
    text = ' '.join(raw_strings)
    iocs = {}
    for ioc_type, pat in IOC_PATTERNS.items():
        hits = list(set(pat.findall(text)))[:5]
        if hits:
            iocs[ioc_type] = hits

    # Threat categorisation
    categories = []
    if all_apis & INJECTION_APIS:   categories.append('Process Injection')
    if all_apis & CRYPTO_APIS:      categories.append('Ransomware/Encryption')
    if all_apis & PERSISTENCE_APIS: categories.append('Persistence')
    if all_apis & NETWORK_APIS:     categories.append('C2 Communication')
    if 'ransom' in text.lower() or 'encrypted' in text.lower():
        if 'Ransomware/Encryption' not in categories:
            categories.append('Ransomware (string match)')
    if 'bitcoin' in text.lower():
        categories.append('Extortion')

    # Risk score
    risk = 0
    risk += len(susp_apis) * 10
    risk += len(susp_strings) * 5
    risk += len(categories) * 15
    risk += len(iocs.get('url', [])) * 5
    risk = min(100, risk)

    return StaticReport(
        sample_name=name, md5=md5, sha256=sha256, is_pe=is_pe,
        extracted_strings=raw_strings[:10],
        suspicious_strings=susp_strings,
        api_imports=list(all_api_names & all_apis),
        suspicious_apis=susp_apis,
        iocs=iocs,
        threat_categories=categories,
        risk_score=risk
    )

reports = [
    static_analyse('SampleA', SAMPLE_A),
    static_analyse('SampleB', SAMPLE_B),
    static_analyse('SampleC', SAMPLE_C),
]

for r in reports:
    print(f'\n=== {r.sample_name} ===')
    print(f'  Risk Score:   {r.risk_score}/100')
    print(f'  Categories:   {r.threat_categories}')
    print(f'  Susp APIs:    {r.suspicious_apis}')
    print(f'  IoCs:         {r.iocs}')

## YARA Rule Generation
YARA rules match byte patterns, string literals, and Boolean logic across file contents. Well-crafted YARA rules have high precision and run against millions of files per second.

In [ ]:
def generate_yara_rule(report: StaticReport) -> str:
    rule_name = 'mal_' + report.sample_name.lower().replace(' ','_')
    strings_section = []
    idx = 0
    for s in report.suspicious_strings[:5]:
        clean = s.replace('\\', '\\\\').replace('"', '\\"')
        strings_section.append(f'        $s{idx} = "{clean}" ascii wide')
        idx += 1
    for api in report.suspicious_apis[:4]:
        strings_section.append(f'        $api{idx} = "{api}" ascii')
        idx += 1
    if not strings_section:
        return '// No significant strings found for YARA rule'
    cond_parts = []
    if report.suspicious_strings:
        n_str = min(2, len(report.suspicious_strings))
        cond_parts.append(f'{n_str} of ($s*)')
    if report.suspicious_apis:
        cond_parts.append('1 of ($api*)')
    condition = ' and '.join(cond_parts) if cond_parts else 'all of them'

    cats = ', '.join(report.threat_categories) if report.threat_categories else 'malware'
    rule = (
        f'rule {rule_name}\n'
        f'{{\n'
        f'    meta:\n'
        f'        description = "Detects {cats}"\n'
        f'        risk_score  = "{report.risk_score}"\n'
        f'        md5         = "{report.md5}"\n'
        f'    strings:\n'
        + '\n'.join(strings_section) + '\n'
        f'    condition:\n'
        f'        uint16(0) == 0x5A4D and\n'  # MZ header
        f'        ({condition})\n'
        f'}}'
    )
    return rule

for r in reports:
    if r.risk_score > 20:
        print(generate_yara_rule(r))
        print()